# 第 4 课：STFT 与声谱图

这一课解决一个关键问题：普通 FFT 告诉我们有哪些频率，但那些频率在什么时候出现？

路线：变化的声音 → 整段 FFT 的局限 → 分帧加窗 → 每帧 FFT → 堆叠成 STFT → 读取声谱图 → 参数交互 → 真实语音。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 声音与声学特征 |
| 建议投入 | 2～4 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 3 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | STFT、声谱图、时间—频率分辨率 |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：STFT、声谱图、时间—频率分辨率。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


<!-- course-bridge-v3 -->
## 知识接力：先取回旧知识，再进入本课

### 3 分钟闭卷回忆

在新 Markdown cell 中回答，**不要先翻前文**：frame/hop 的索引；窗函数的目的；单帧 FFT 的频率轴。

- 三项都能用“含义 + 单位/shape + 一个数字例子”回答：进入本课。
- 能回答两项：学习本课，但把缺口记入 `LEARNING_LOG.md`。
- 只能回答零到一项：先回到 [主线第 2～3 课](核心课程索引_第01到41课.md)，做一次最小实验；不要靠继续看新术语掩盖断点。

### 本课接口契约

```text
输入：长波形、frame/window/hop、n_fft
  ↓ 本课要学会的变换、状态或判断
输出：时间帧 × 频率 bin 的功率谱及两条坐标轴
```

学完后必须能解释：输入的哪个单位/shape/状态若丢失，会让输出“仍能运行却语义错误”。


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import ipywidgets as widgets
from IPython.display import Audio, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

## 1. 为什么整段 FFT 不够？

我们制造两个 1 秒信号：A 先 300 Hz 后 1000 Hz；B 先 1000 Hz 后 300 Hz。它们的时间顺序不同，但整段频谱几乎相同。

In [ ]:
sr = 8000
half_t = np.arange(sr//2)/sr
tone300 = np.sin(2*np.pi*300*half_t)
tone1000 = np.sin(2*np.pi*1000*half_t)
signal_a = np.concatenate([tone300, tone1000])
signal_b = np.concatenate([tone1000, tone300])
time_axis = np.arange(sr)/sr

fig, axes = plt.subplots(2, 2, figsize=(13, 6))
for row, (signal, name) in enumerate([(signal_a,'A: 300 → 1000 Hz'),(signal_b,'B: 1000 → 300 Hz')]):
    axes[row,0].plot(time_axis, signal, linewidth=0.5)
    axes[row,0].set(title=name, xlabel='Time (s)', ylabel='Amplitude')
    spec = np.abs(np.fft.rfft(signal*np.hanning(len(signal))))
    freqs = np.fft.rfftfreq(len(signal), 1/sr)
    axes[row,1].plot(freqs, spec)
    axes[row,1].set_xlim(0,1500); axes[row,1].set(title='整段频谱',xlabel='Frequency (Hz)',ylabel='Magnitude')
for ax in axes.ravel(): ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

### 思考题 1

1. A 和 B 的发声顺序相同吗？
2. 两张整段频谱是否都在 300 Hz、1000 Hz 附近出现峰？
3. 只看整段频谱，能判断哪个频率先出现吗？

结论：整段 FFT 汇总了整段音频的频率成分，却丢失了出现时间。

## 2. STFT 的核心想法

STFT = Short-Time Fourier Transform，短时傅里叶变换：

1. 用 frame length 切出短帧。
2. 相邻帧相隔 hop length。
3. 每帧乘 window。
4. 每帧做 FFT。
5. 按时间顺序堆叠每帧频谱。

结果是二维矩阵：`时间帧 × 频率 bin`。画成图后，横轴时间、纵轴频率、颜色表示能量。

## 3. 亲手实现 STFT：第一步，切帧

下面不调用现成的 STFT 函数，确保每一步都看得见。

In [ ]:
def make_frames(audio, frame_size, hop_size):
    starts = np.arange(0, len(audio)-frame_size+1, hop_size)
    frames = np.stack([audio[s:s+frame_size] for s in starts])
    return frames, starts

frame_ms = 25
hop_ms = 10
frame_size = round(sr*frame_ms/1000)
hop_size = round(sr*hop_ms/1000)
frames, starts = make_frames(signal_a, frame_size, hop_size)
print('音频形状:', signal_a.shape)
print('frames 形状:', frames.shape)
print('帧数:', len(frames))
print('每帧采样点:', frames.shape[1])
print('帧移采样点:', hop_size)

`frames.shape=(帧数, 每帧采样点)`。这里每一行是一帧，不是一个频率。下一步才把每行从时域变到频域。

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 6), sharex=True)
for ax, index in zip(axes, [10, 45, 80]):
    local_t = np.arange(frame_size)/sr*1000
    ax.plot(local_t, frames[index])
    ax.set_title(f'frame {index}，开始时间 {starts[index]/sr:.3f} s')
    ax.set_ylabel('Amplitude'); ax.grid(alpha=0.25)
axes[-1].set_xlabel('Time inside frame (ms)')
plt.tight_layout(); plt.show()

### 练习 1

1. 8 kHz、25 ms 一帧有多少采样点？
2. 8 kHz、10 ms hop 有多少采样点？
3. frame 10、45、80 分别大约位于音频的前部、中部还是后部？

## 4. 第二步：加窗并对每帧做 FFT

`axis=1` 非常重要：对每一行，也就是每一帧做 FFT。

In [ ]:
n_fft = 256
window = np.hanning(frame_size)
windowed_frames = frames * window[None, :]
stft_complex = np.fft.rfft(windowed_frames, n=n_fft, axis=1)
power = np.abs(stft_complex)**2
freqs = np.fft.rfftfreq(n_fft, 1/sr)
frame_times = starts/sr
print('加窗后 frames:', windowed_frames.shape)
print('复数 STFT:', stft_complex.shape)
print('功率谱:', power.shape)
print('非负频率 bin:', len(freqs))

为什么是 129 个频率 bin？因为 256 点实数 FFT 保留 $256/2+1=129$ 个非负频率：包含 0 Hz 和 4000 Hz。

## 5. 第三步：画声谱图

矩阵 `power` 的形状是 `(时间帧, 频率 bin)`，绘图时转置为 `(频率, 时间)`。

In [ ]:
power_db = 10*np.log10(np.maximum(power, 1e-12))
power_db -= power_db.max()  # 最强点设为 0 dB
plt.figure(figsize=(12, 5))
image = plt.pcolormesh(frame_times, freqs, power_db.T, shading='auto', cmap='magma', vmin=-80, vmax=0)
plt.axvline(0.5, color='cyan', linestyle='--', label='频率切换位置')
plt.xlabel('Time (s)'); plt.ylabel('Frequency (Hz)')
plt.title('Signal A 的 STFT spectrogram')
plt.ylim(0,1500); plt.colorbar(image,label='Relative power (dB)')
plt.legend(); plt.show()

读图：0～0.5 秒在约 300 Hz 有亮横线；0.5～1 秒在约 1000 Hz 有亮横线。STFT 找回了整段 FFT 丢失的时间顺序。颜色条中 0 dB 是全图最强点，-20 dB 表示功率比最强点低 100 倍。

### 练习 2：读声谱图

1. 横轴、纵轴、颜色分别表示什么？
2. 300 Hz 在什么时候出现？
3. 1000 Hz 在什么时候出现？
4. 0 dB 是否表示现实世界完全没有声音？

## 6. 对比 A 与 B：时间顺序终于可见


In [ ]:
def simple_stft(audio, sr, frame_ms=25, hop_ms=10, n_fft=256):
    frame_n = round(sr*frame_ms/1000)
    hop_n = round(sr*hop_ms/1000)
    frames, starts = make_frames(audio, frame_n, hop_n)
    z = np.fft.rfft(frames*np.hanning(frame_n)[None,:], n=n_fft, axis=1)
    p = np.abs(z)**2
    db = 10*np.log10(np.maximum(p,1e-12)); db -= db.max()
    return db, starts/sr, np.fft.rfftfreq(n_fft,1/sr)

fig, axes = plt.subplots(2,1,figsize=(12,7),sharex=True,sharey=True)
for ax, sig, title in zip(axes,[signal_a,signal_b],['A: 300 → 1000 Hz','B: 1000 → 300 Hz']):
    db, tt, ff = simple_stft(sig,sr)
    im=ax.pcolormesh(tt,ff,db.T,shading='auto',cmap='magma',vmin=-80,vmax=0)
    ax.set_title(title); ax.set_ylabel('Frequency (Hz)'); ax.set_ylim(0,1500)
axes[-1].set_xlabel('Time (s)'); fig.colorbar(im,ax=axes,label='Relative power (dB)')
plt.show()

## 7. 交互实验：frame 与 hop 怎样改变声谱图？

拖动帧长：短帧更擅长定位时间变化，长帧更擅长区分接近的频率。拖动 hop：主要改变横向采样密度。

In [ ]:
def draw_stft_parameters(frame_ms=25,hop_ms=10,n_fft=256):
    db,tt,ff=simple_stft(signal_a,sr,frame_ms,hop_ms,n_fft)
    plt.figure(figsize=(11,4))
    im=plt.pcolormesh(tt,ff,db.T,shading='auto',cmap='magma',vmin=-80,vmax=0)
    plt.ylim(0,1500); plt.xlabel('Time (s)'); plt.ylabel('Frequency (Hz)')
    plt.title(f'frame={frame_ms} ms, hop={hop_ms} ms, n_fft={n_fft}, shape={db.shape}')
    plt.colorbar(im,label='Relative power (dB)'); plt.show()

widgets.interact(
    draw_stft_parameters,
    frame_ms=widgets.IntSlider(value=25,min=10,max=80,step=5,description='帧长'),
    hop_ms=widgets.IntSlider(value=10,min=5,max=30,step=5,description='帧移'),
    n_fft=widgets.Dropdown(options=[128,256,512,1024],value=256,description='n_fft')
)

### 交互题 A

1. frame 从 10 ms 增加到 80 ms，0.5 秒的切换边界更清晰还是更模糊？
2. hop 从 5 ms 增加到 30 ms，帧数增加还是减少？
3. n_fft 增大后频率方向像素变多，是否代表原始声音信息凭空增加？
4. 若 frame 比 n_fft 还长，当前函数可能出错或截断。为什么通常要求 `n_fft >= frame_samples`？

## 8. 为什么使用 dB，而不是直接画线性功率？

语音能量范围很大。线性图常被最强区域支配，弱结构看不清；对数 dB 压缩动态范围。

In [ ]:
fig,axes=plt.subplots(2,1,figsize=(12,7),sharex=True,sharey=True)
im1=axes[0].pcolormesh(frame_times,freqs,power.T,shading='auto',cmap='magma')
axes[0].set_title('线性功率')
im2=axes[1].pcolormesh(frame_times,freqs,power_db.T,shading='auto',cmap='magma',vmin=-80,vmax=0)
axes[1].set_title('相对 dB：弱结构更容易看见')
for ax in axes: ax.set_ylim(0,1500); ax.set_ylabel('Frequency (Hz)')
axes[1].set_xlabel('Time (s)')
fig.colorbar(im1,ax=axes[0],label='Power'); fig.colorbar(im2,ax=axes[1],label='dB')
plt.show()

## 9. 分析并播放真实语音

先播放数字 0，再观察其波形和 STFT。

In [ ]:
audio_path=ROOT/'data'/'0_jackson_0.wav'
audio,audio_sr=sf.read(audio_path)
if audio.ndim>1: audio=audio.mean(axis=1)
display(Audio(audio,rate=audio_sr))
real_db,real_times,real_freqs=simple_stft(audio,audio_sr,25,10,256)
real_time=np.arange(len(audio))/audio_sr
fig,axes=plt.subplots(2,1,figsize=(12,7),sharex=True)
axes[0].plot(real_time,audio,linewidth=0.7); axes[0].set(title='真实语音波形',ylabel='Amplitude')
im=axes[1].pcolormesh(real_times,real_freqs,real_db.T,shading='auto',cmap='magma',vmin=-80,vmax=0)
axes[1].set(title='真实语音 STFT',xlabel='Time (s)',ylabel='Frequency (Hz)')
fig.colorbar(im,ax=axes[1],label='Relative power (dB)')
plt.tight_layout(); plt.show()

### 真实语音看图题

1. 波形振幅最大的时间区域，在声谱图上是否也通常较亮？
2. 人声是否只有一条频率线？
3. 声谱图中的多条横向纹理可能来自什么？提示：基频、谐波、共振峰。
4. 为什么真实语音比两个纯音复杂得多？

## 10. STFT 形状计算

不考虑 padding，帧数大约为：

$$T=1+\left\lfloor\frac{L-frame\_size}{hop\_size}\right\rfloor$$

非负频率 bin 数为：$F=n\_fft/2+1$。因此 STFT 形状是 $(T,F)$。

In [ ]:
def shape_calculator(audio_seconds=1.0,sample_rate=16000,frame_ms=25,hop_ms=10,n_fft=512):
    L=round(audio_seconds*sample_rate)
    frame_n=round(frame_ms*sample_rate/1000)
    hop_n=round(hop_ms*sample_rate/1000)
    T=1+(L-frame_n)//hop_n if L>=frame_n else 0
    F=n_fft//2+1
    print(f'采样点 L={L}, frame={frame_n}, hop={hop_n}')
    print(f'STFT shape = ({T}, {F})')
widgets.interact(shape_calculator,
    audio_seconds=widgets.FloatSlider(value=1,min=0.1,max=5,step=0.1,description='秒数'),
    sample_rate=widgets.Dropdown(options=[8000,16000,44100],value=16000,description='采样率'),
    frame_ms=widgets.IntSlider(value=25,min=10,max=50,step=5,description='帧长'),
    hop_ms=widgets.IntSlider(value=10,min=5,max=30,step=5,description='帧移'),
    n_fft=widgets.Dropdown(options=[256,512,1024],value=512,description='n_fft'))

## 本课测试

1. 整段 FFT 为什么无法区分 300→1000 Hz 与 1000→300 Hz？
2. STFT 的五个基本步骤是什么？
3. STFT 矩阵的两个维度分别表示什么？
4. 声谱图的横轴、纵轴和颜色分别表示什么？
5. 8 kHz、25 ms、10 ms hop 分别是多少采样点？
6. 256 点实数 FFT 有多少个非负频率 bin？
7. 为什么画图前常把功率转换成 dB？
8. 相对 dB 图中的 0 dB 表示什么？
9. frame 变长时，时间分辨率和频率分辨率怎样变化？
10. hop 变小时，时间帧数怎样变化？
11. 1 秒、16 kHz、25 ms frame、10 ms hop、不 padding 时大约多少帧？
12. 为什么真实人声的声谱图有复杂纹理？

参考答案：1 整段频谱汇总频率而不保留出现时间；2 分帧、加窗、每帧 FFT、取幅度/功率、按时间堆叠；3 时间帧和频率 bin；4 时间/频率/能量；5 200/80；6 129；7 压缩动态范围；8 全图最强时频点；9 时间分辨率下降、频率分辨率提高；10 增加；11 约 98 帧；12 人声包含随时间变化的基频、谐波、共振峰和噪声。

## 小结

STFT 把频谱从一张静态清单变成随时间变化的地图：

`waveform → frames → window → FFT per frame → power → dB → spectrogram`

下一课：Mel。我们会看到人耳为什么不按线性 Hz 感知频率，以及三角 Mel 滤波器如何把 257 个 FFT bin 压缩成 40 或 80 个 Mel bin。

<!-- course-upgrade-v2 -->
## 强化练习：第 4 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `STFT`、`声谱图`、`时间—频率分辨率`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**窗长增大但 hop 不变**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**手写 STFT 并重建时间/频率坐标**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**连接第 3 课单帧频谱与整段声谱图**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：STFT、声谱图、时间—频率分辨率。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 STFT、声谱图、时间—频率分辨率。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
